[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Jibby2k1/SPS_Curriculum/blob/main/Intro_Math/Random_Matrix_Theory/Random_Matrix_Theory.ipynb)


**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Random Matrix Theory

What do the eigenvalues of a *random* matrix look like? Not random at all — they obey laws as sharp as the CLT, and those laws decide when [covariance estimation](../../Intro_DSP/Statistical_Signal_Processing.ipynb), [MUSIC](../../Intro_DSP/Array_Processing.ipynb), and PCA can be trusted. Three sessions: the semicircle, Marchenko–Pastur (with the analytic edges verified), and the spiked-model detection threshold — the phase transition every array processor should know by heart.

## 1. Pre-requisites

[Linear Algebra](../Linear_Algebra/Linear_Algebra.ipynb) S3–S4; [Concentration](../Concentration/Concentration_Inequalities.ipynb) for the 'why so deterministic' intuition.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(0)

---
### 🕐 Session 1 of 3 — *The Semicircle Law* (~35 min)
**Goal:** eigenvalues of symmetric random matrices: individually random, collectively deterministic.
**Builds on:** [Linear Algebra](../Linear_Algebra/Linear_Algebra.ipynb) S3. &nbsp; **Feeds into:** Session 2 (Marchenko–Pastur).

---

## 2. Order from Chaos

💡 **Intuition.** Fill a symmetric matrix with i.i.d. noise, scale by $1/\sqrt{n}$, and its eigenvalue *histogram* converges to a fixed shape — Wigner's semicircle — with no randomness left in the limit. Same magic as the [LLN](../Analysis/Independence.ipynb): each eigenvalue depends on *all* $n^2/2$ entries, no single entry matters ([McDiarmid!](../Concentration/Concentration_Inequalities.ipynb)), so the ensemble self-averages. Eigenvalues also *repel* each other — near-collisions are rare — which is why the histogram is smooth, not clumpy.

In [ ]:

# YOUR CODE HERE


**What just happened.** **One** random matrix — not an average over many — and its eigenvalue histogram already lies on Wigner's semicircle. Largest eigenvalue **1.993** against a theoretical edge of exactly 2.0, and the fraction of eigenvalues outside $[-2,2]$ is **0.0000**.

**The paradox and its resolution.** Every entry was drawn independently at random; nothing was designed. Yet the *collective* behaviour of the eigenvalues has no randomness left in it. The reason is that a single $2000 \times 2000$ symmetric matrix already contains about two million independent numbers, and **each eigenvalue depends on all of them** while being insensitive to any one. That is exactly the bounded-differences condition behind [McDiarmid's inequality](../Concentration/Concentration_Inequalities.ipynb), so the spectrum concentrates — the [law of large numbers](../Analysis/Independence.ipynb) acting on a matrix rather than on a sequence.

The practical consequence is worth naming: **one draw is already an ensemble.** You do not need to average over many random matrices to see the law, which is why this demo works at all and why random matrix predictions are usable on single real datasets.

**Note the $1/\sqrt{n}$ scaling.** Without it the eigenvalues would grow like $\sqrt{n}$ and no limiting shape would exist. It is the same normalisation that makes the central limit theorem well-posed, playing the same role for the same reason.

**And note that the edge is hard.** Not one eigenvalue in 2000 strayed outside $[-2,2]$, and the maximum came within 0.007 of the boundary. The semicircle does not have tails that fade out — it *stops*. That sharpness is what makes Sessions 2 and 3 useful: if the noise had soft tails, "is this eigenvalue too big to be noise?" would be a matter of degree. Because the edge is hard, it becomes a genuine threshold.

**One phenomenon the histogram hides.** Eigenvalues of a random symmetric matrix *repel* one another — near-collisions are far rarer than for independent random points, which is why the histogram is smooth rather than clumpy. This local behaviour turns out to match the spacing statistics of nuclear energy levels and, empirically, the zeros of the Riemann zeta function. Not needed for what follows, but it is a fair indication that these laws are deeper than a convenience for signal processing.

---
### 🕐 Session 2 of 3 — *Marchenko–Pastur: the Law of Sample Covariance* (~40 min)
**Goal:** what eigenvalues of pure-noise covariance look like — and why high-dimensional PCA lies.
**Builds on:** Session 1. &nbsp; **Feeds into:** Session 3 (spiked models).

---

## 3. The Noise Bulk

💡 **Intuition.** Estimate a covariance from $n$ samples of $p$-dimensional *white* noise (true covariance $= I$: all eigenvalues 1). With $p/n = \gamma$ not small, the sample eigenvalues **spread** across $[(1-\sqrt\gamma)^2, (1+\sqrt\gamma)^2]$ — the Marchenko–Pastur bulk. At $p/n = 1/2$, 'eigenvalues' of pure noise range from 0.09 to 2.9! Every PCA scree plot with $p \sim n$ contains this artifact, and everything inside the bulk is *structurally indistinguishable from noise*.

In [ ]:
# ORACLE: empirical bulk edges vs the analytic (1 ± √γ)²   [γ = p/n]

# YOUR CODE HERE


**What just happened.** The true covariance is the **identity** — every true eigenvalue is exactly 1. The *sample* covariance's eigenvalues fill $[0.087, 2.927]$, matching the analytic Marchenko–Pastur edges $(0.0858, 2.9142)$ to two decimal places, with the `assert` enforcing agreement to 0.05.

**Read that again, because it is alarming.** From data with perfectly isotropic truth, we obtained a spread of more than **30×** between the smallest and largest sample eigenvalue. A scree plot of this data would show a commanding "first component" at 2.9 and a "negligible" direction at 0.09. **Both are artifacts.** There is no structure whatsoever in the underlying distribution.

**Why classical intuition fails here.** We estimated $p(p+1)/2 \approx 500{,}000$ covariance parameters from $n \times p = 2$ million numbers — about four observations per parameter. Classical asymptotics assume $n \to \infty$ with $p$ *fixed*, which is simply not the regime we are in. Random matrix theory is the correct asymptotic when $p$ and $n$ grow together, and the governing parameter is $\gamma = p/n$. Classical statistics is not wrong; it is answering a different question.

**The edge formula is the thing to memorise:** $[(1-\sqrt\gamma)^2, (1+\sqrt\gamma)^2]$. Evaluate it at a few ratios and "high-dimensional" stops being vague:

| $\gamma = p/n$ | noise bulk |
|---|---|
| 0.01 | [0.81, 1.21] |
| 0.25 | [0.25, 2.25] |
| 0.50 | [0.09, 2.91] |
| 1.00 | [0.00, 4.00] |

The problem is not that $p$ is large. It is that $p/n$ is not small. Ten thousand samples of a thousand-dimensional variable is $\gamma = 0.1$ and still visibly spread.

**Which gives you an immediately usable check.** Before believing any principal component, compute $(1+\sqrt{p/n})^2$ for your data and ask whether the eigenvalue exceeds it. **Anything inside the bulk is structurally indistinguishable from noise**, no matter how dominant it looks relative to its neighbours. Most practitioners have never applied this test, and it invalidates a meaningful fraction of published scree-plot interpretation in genomics, finance, and neuroimaging.

Closest to home, this is the quantitative form of "how many snapshots does [MUSIC](../../Intro_DSP/Array_Processing.ipynb) need?" — enough that $\gamma$ is small enough for real sources to clear the bulk. Session 3 makes that a sharp threshold.

---
### 🕐 Session 3 of 3 — *Spiked Models & the Detection Threshold* (~40 min)
**Goal:** when does a real signal's eigenvalue escape the noise bulk? The BBP phase transition.
**Builds on:** Session 2.

---

## 4. The Phase Transition

💡 **Intuition.** Add one rank-one signal of strength $\theta$ to the noise (a source hitting an [array](../../Intro_DSP/Array_Processing.ipynb)). Does the top sample eigenvalue reveal it? **Only above a threshold**: for $\theta > \sqrt{\gamma}$ the top eigenvalue pops out of the bulk at $(1+\theta)(1+\gamma/\theta)$; below it, the spike is *swallowed* — no eigenvalue method can see it, however clever (the BBP transition). This is the sharp version of 'how many snapshots do I need': MUSIC's source count, PCA's component count, all gated by $\theta \gtrless \sqrt{p/n}$.

In [ ]:
# sweep the spike strength through the threshold — ORACLE: the BBP position formula

# YOUR CODE HERE


**What just happened.** Sweeping the spike strength $\theta$ through the threshold $\sqrt\gamma = 0.5$ produces a **phase transition**, not a gradual improvement. Below it the top eigenvalue sits flat at the bulk edge; above it, it detaches and tracks the BBP prediction $(1+\theta)(1+\gamma/\theta)$.

**The right-hand panel is the one that matters, and its number needs a baseline.** Below threshold the eigenvector overlap with the true direction is **0.034**. That sounds small; the question is *how* small. For a random unit vector in $p = 400$ dimensions, the expected overlap with any fixed direction is $\sqrt{2/\pi p} = \mathbf{0.040}$.

So 0.034 is not "weak recovery" — it is **exactly the random-guess floor**. The estimated eigenvector below threshold carries no information about the signal whatsoever. It is not a degraded answer to be improved with better processing; it is a vector pointing nowhere in particular, and reporting it as a direction estimate would be reporting noise.

**Why this is a threshold and not a fade.** Above $\sqrt\gamma$ the signal's eigenvalue escapes the noise bulk and becomes visible; below it, the signal's eigenvalue is *inside* the bulk, where Session 2 established that everything is structurally indistinguishable from noise. There is no intermediate regime where detection is difficult but possible by spectral means. The transition is sharp — which is a real surprise for anyone trained to expect that more SNR is gradually better.

**Turn the threshold into a snapshot budget, because that is what it is for.** $\theta > \sqrt{p/n}$ rearranges to
$$n > \frac{p}{\theta^2}.$$
A 100-element array with a spike of strength 0.5 needs more than 400 snapshots. That is the quantitative version of the usual hand-wave that "[MUSIC](../../Intro_DSP/Array_Processing.ipynb) needs enough snapshots," and it is checkable before you build anything. The same inequality governs how many observations a factor model needs before its factors are real.

**Be honest about the fit.** Maximum deviation from BBP above threshold is **0.191** — not tight. Each point is a single draw at $p = 400$, and BBP is an asymptotic law, so finite-size fluctuation dominates; averaging over draws or increasing $p$ would shrink it considerably. The verified claim here is the *shape* — flat below threshold, detaching above, following the predicted curve — rather than pointwise agreement.

**And the consequence worth carrying away.** Below threshold, **no eigenvalue method can see the signal.** Not MUSIC, not PCA, not any more sophisticated subspace algorithm — because the information is genuinely absent from the spectrum rather than merely hard to extract. Recovering it requires bringing in structure the spectrum does not use: sparsity ([compressed sensing](../../Intro_DSP/Compressed_Sensing.ipynb)), a known waveform (matched filtering), or temporal correlation. Knowing where the wall is tells you when to stop tuning your eigen-solver and start changing your assumptions.

## 5. Conclusion

Random eigenvalues obey deterministic laws; pure noise fills a predictable bulk (edges verified to 2 decimals); and signals are detectable by spectra *only* above $\sqrt{p/n}$ — a phase transition, not a gradual fade. Check every scree plot against Marchenko–Pastur before believing a single 'component'.

---
## Where next

- [Array Processing](../../Intro_DSP/Array_Processing.ipynb) — MUSIC's snapshot budget, now quantitative.
- [Numerical Linear Algebra](../Numerical_Linear_Algebra/Numerical_Linear_Algebra.ipynb) — why random sketches capture ranges.
- [Concentration](../Concentration/Concentration_Inequalities.ipynb) — the self-averaging machinery.